# V0.4 Context VM Lab

问题：跑 50 轮后 Session 很大，为什么不全部塞给模型？

这个 notebook 逐格展示 durable Session truth 与 bounded model-visible working set 的区别。

## Mode

`deterministic`：Model decision = SCRIPTED，Kernel execution = REAL。

`real_model`：Model decision = REAL OpenAI-compatible，Kernel execution = REAL。优先使用 `AGENTKERNEL_LAB_LLM_*`，否则复用仓库本地 `.minicode/config.json`。不会展示 hidden chain-of-thought，也不会展示 API key。

In [ ]:
MODE = "deterministic"

from pathlib import Path
import sys

def find_agentkernel_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir() and (path / "labs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the AgentKernel repo root or the labs directory.")

REPO_ROOT = find_agentkernel_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from labs import create_lab

lab = create_lab("v04", mode=MODE)


## Step 1: Setup

创建一个有 50 轮历史的 Session。注意这些事件是 durable truth，不等于下一次模型输入。

In [ ]:
lab.setup()

## Step 2: Inspect durable truth

先看完整 Session：它保留所有历史消息和事件。

In [ ]:
lab.show_session_truth()

## Step 3: Build bounded working set

Context VM 从完整 Session 投影出适合当前模型调用的 working set。

In [ ]:
lab.build_working_set()

## Step 4: Inspect model-visible request

现在看模型实际能看到的 messages；这应小于完整 Session history。

In [ ]:
lab.show_model_request()

## Step 5: Ask deterministic or real model

`deterministic` 模式用脚本响应；`real_model` 模式会调用本地配置的大模型，但仍只展示可观察输出。

In [ ]:
lab.model_step()

## Summary

这个实验回答：Context 是投影，不是事实源。

In [ ]:
lab.summary()
lab.close()